In [11]:
import pandas as pd
import feature_engineering_helper as hf
import importlib
from sklearn.model_selection import StratifiedShuffleSplit

In [12]:
# get raw data
data_prefix = '../data_curation/processed_data/'
original_train = pd.read_csv(data_prefix + 'train_without_data_augmentation.csv')[['SMILES','MP']]

original_test = pd.read_csv(data_prefix + 'test_predictions.csv')[['SMILES','exp MP']]
original_test = original_test.rename(columns={'exp MP': 'MP'})

raw_data = pd.concat([original_train, original_test], axis=0).reset_index(drop=True)

print(original_train.shape)
print(original_test.shape)
print(raw_data.shape)

(17633, 2)
(1961, 2)
(19594, 2)


In [13]:
# Clean data with new melting point bounds (between low_bound and high_bound)
clean_data = hf.clean_dataset(raw_data, low_bound=0, high_bound=500)

Initial shape: (19594, 2)
Removed 2370 rows with MP < 0
Removed 2 rows with MP > 500
Final shape: (17222, 2)


In [14]:
# add Ro5 labels
importlib.reload(hf)
clean_data['Ro5'] = clean_data['SMILES'].apply(hf.check_Ro5)
# how many molecules in percentage violate Ro5
violation_percentage = 100 * (clean_data['Ro5'] == False).mean()
print(f"{violation_percentage:.2f}% of molecules violate Ro5")
clean_data.describe()

2.13% of molecules violate Ro5


,MP,Ro5
count,17222.000000,17222.00000
mean,126.045995,0.97869
std,70.913439,0.14442
min,0.000000,0.00000
25%,69.000000,1.00000
50%,120.000000,1.00000
75%,175.000000,1.00000
max,492.500000,1.00000


In [15]:
data_with_features = hf.smiles_to_features(clean_data, ['rdkit'])
data_with_features

✓ RDKit: Added 217 features


/Users/sdl5_mp/Documents/GitHub/melting_point_2026/Ro5/feature_engineering_helper.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_with_features[f'RDKit_{name}'] = [f[i] for f in rdkit_features]
/Users/sdl5_mp/Documents/GitHub/melting_point_2026/Ro5/feature_engineering_helper.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_with_features[f'RDKit_{name}'] = [f[i] for f in rdkit_features]
/Users/sdl5_mp/Documents/GitHub/melting_point_2026/Ro5/feature_engineering_helper.py:43: PerformanceWarning: DataFrame is high

,MP,RDKit_AvgIpc,RDKit_BCUT2D_CHGHI,RDKit_BCUT2D_CHGLO,RDKit_BCUT2D_LOGPHI,RDKit_BCUT2D_LOGPLOW,RDKit_BCUT2D_MRHI,RDKit_BCUT2D_MRLOW,RDKit_BCUT2D_MWHI,RDKit_BCUT2D_MWLOW,...,RDKit_fr_term_acetylene,RDKit_fr_tetrazole,RDKit_fr_thiazole,RDKit_fr_thiocyan,RDKit_fr_thiophene,RDKit_fr_unbrch_alkane,RDKit_fr_urea,RDKit_qed,Ro5,SMILES
0,0.0,2.021826,1.969209,-1.950422,1.839529,-2.030578,5.910992,-0.139527,16.536981,10.480196,...,0,0,0,0,0,0,0,0.461644,1,CCOC(=O)/C=C/C(=O)OCC
1,230.0,2.596470,2.134805,-2.073420,2.259390,-2.109867,6.349561,0.072022,35.499021,10.129954,...,0,0,0,0,0,0,0,0.322574,1,O=C(c1ccc(cc1)C(=O)Oc1cc(Cl)cc(c1)Cl)Oc1cc(Cl)...
2,285.0,2.841923,2.064669,-2.060442,2.237060,-2.108735,6.045732,0.101400,16.151178,10.174814,...,0,0,0,0,0,0,0,0.344216,1,O=C(c1ccccc1)Nc1cccc(c1)/N=N/c1cccc(c1)NC(=O)c...
4,112.0,2.046001,1.922596,-1.962809,1.975790,-1.933211,5.089489,0.265376,16.253948,10.295277,...,0,0,0,0,0,0,0,0.609080,1,OCc1cccc(n1)CO
5,160.0,3.160349,2.275297,-2.119670,2.398983,-2.132280,7.992453,0.414605,35.495692,10.179599,...,0,0,0,0,0,0,0,0.779142,1,COc1ccc(cc1)c1nnc2n1NC(S2)c1ccc(cc1)Cl
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19588,49.0,1.956738,2.043838,-1.965025,2.082921,-2.002395,5.695900,-0.135729,19.159738,10.139909,...,0,0,0,0,0,0,0,0.735003,1,OC(=O)Cc1ccc(c(c1)F)F
19589,72.0,2.416826,2.177119,-2.214041,2.250524,-2.260984,5.953675,0.162449,16.550348,10.219639,...,0,0,0,0,0,0,0,0.812970,1,CCOC(=O)N(c1ccccc1)c1ccccc1
19591,90.0,1.956738,2.023046,-1.983385,2.220807,-1.989227,6.415186,-0.135725,35.498261,10.149605,...,0,0,0,0,0,0,0,0.805020,1,OC(=O)Cc1ccc(c(c1)Cl)Cl
19592,148.0,2.213081,2.134795,-2.053354,2.349259,-2.050401,5.905731,0.050385,16.532856,10.160549,...,0,0,0,0,0,0,0,0.519957,1,CCCOC(=O)c1cc(O)c(c(c1)O)O


In [16]:
# drop rows with NaN values and print how many rows were dropped and reset index
nan_rows = data_with_features[data_with_features.isna().any(axis=1)]
num_dropped = nan_rows.shape[0]
print(f"Dropped {num_dropped} rows containing NaN values.")
data_with_features = data_with_features.dropna().reset_index(drop=True)

Dropped 2 rows containing NaN values.


In [18]:
# stratified split based on Ro5 compliance

split = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
for train_index, test_index in split.split(data_with_features, data_with_features['Ro5']):
    strat_train_set = data_with_features.loc[train_index]
    strat_test_set = data_with_features.loc[test_index]
    strat_train_set['Type'] = 'Train'
    strat_test_set['Type'] = 'Test'
final_data = pd.concat([strat_train_set, strat_test_set], axis=0).reset_index(drop=True)

# print the shape of train and test dataset
print(f"Train set: {final_data[final_data['Type'] == 'Train'].shape}")
print(f"Train set: {final_data[final_data['Type'] == 'Test'].shape}")

# print the percentage of Ro5 violations in train and test sets
train_violation_percentage = 100 * (final_data[final_data['Type'] == 'Train']['Ro5'] == False).mean()
test_violation_percentage = 100 * (final_data[final_data['Type'] == 'Test']['Ro5'] == False).mean()
print(f"Train set: {train_violation_percentage:.2f}% of molecules violate Ro5")
print(f"Test set: {test_violation_percentage:.2f}% of molecules violate Ro5")


Train set: (8610, 221)
Train set: (8610, 221)
Train set: 2.13% of molecules violate Ro5
Test set: 2.14% of molecules violate Ro5


In [19]:
final_data.to_parquet(data_prefix + 'data_with_all_features_RDKit_50.parquet', compression= "zstd")